In [4]:
import pandas as pd
import glob
import os

DADOS_DIR = "dados_marilia"
OUTPUT_CSV = os.path.join(DADOS_DIR, "resumo_analise.csv")

In [5]:
# ── Carrega todos os arquivos de dados_marilia ────────────────────────────────
files = sorted(glob.glob(os.path.join(DADOS_DIR, "saida_*.csv")))
print(f"Arquivos encontrados: {len(files)}")

dfs = []
for f in files:
    df = pd.read_csv(f, sep=";")
    dfs.append(df)
    print(f"  {os.path.basename(f)}: {len(df):,} registros")

dados = pd.concat(dfs, ignore_index=True)
print(f"\nTotal de registros: {len(dados):,}")
print(f"Colunas: {list(dados.columns)}")

Arquivos encontrados: 5
  saida_2019.csv: 71,939 registros
  saida_2020.csv: 52,068 registros
  saida_2021.csv: 43,543 registros
  saida_2022.csv: 47,275 registros
  saida_2023.csv: 20,930 registros

Total de registros: 235,755
Colunas: ['ID', 'Local', 'Ano', 'Semana', 'Data Inicio', 'Data Fim', 'Latitude', 'Longitude', 'Aedes aegypti', 'Aedes albopictus', 'Culex sp', 'Precipitation', 'Temperature', 'Relative humidity', 'Mes']


In [6]:
# ── Agrega por semana e exporta ───────────────────────────────────────────────
COLUNAS_SOMA = ["Aedes aegypti", "Aedes albopictus", "Culex sp"]
COLUNAS_GRUPO = ["Ano", "Semana", "Data Inicio", "Data Fim", "Mes"]

resumo = (
    dados
    .groupby(COLUNAS_GRUPO, as_index=False)[COLUNAS_SOMA]
    .sum()
    .sort_values(["Ano", "Semana"])
    .reset_index(drop=True)
)

# ── Cria coluna Mes_editado (ex: JAN1, JAN2, FEV1, ...) ──────────────────────
MES_ABREV = {
    1: "JAN", 2: "FEV", 3: "MAR", 4: "ABR",
    5: "MAI", 6: "JUN", 7: "JUL", 8: "AGO",
    9: "SET", 10: "OUT", 11: "NOV", 12: "DEZ"
}

resumo["Data Inicio"] = pd.to_datetime(resumo["Data Inicio"])
resumo = resumo.sort_values("Data Inicio").reset_index(drop=True)
resumo["_mes_num"] = resumo["Data Inicio"].dt.month
resumo["_rank"] = resumo.groupby(["Ano", "_mes_num"]).cumcount() + 1
resumo["Mes_editado"] = resumo["_mes_num"].map(MES_ABREV) + resumo["_rank"].astype(str)
resumo = resumo.drop(columns=["_mes_num", "_rank"])

resumo.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"CSV salvo em: {OUTPUT_CSV}")
print(f"  Semanas cobertas: {len(resumo):,}")
print(f"  Colunas: {list(resumo.columns)}")
resumo.head(10)

CSV salvo em: dados_marilia\resumo_analise.csv
  Semanas cobertas: 232
  Colunas: ['Ano', 'Semana', 'Data Inicio', 'Data Fim', 'Mes', 'Aedes aegypti', 'Aedes albopictus', 'Culex sp', 'Mes_editado']


,Ano,Semana,Data Inicio,Data Fim,Mes,Aedes aegypti,Aedes albopictus,Culex sp,Mes_editado
0,2019,1,2018-12-30,2019-01-05,12,55,1,128,DEZ1
1,2019,2,2019-01-06,2019-01-12,1,57,10,180,JAN1
2,2019,3,2019-01-13,2019-01-19,1,455,46,628,JAN2
3,2019,4,2019-01-20,2019-01-26,1,371,29,549,JAN3
4,2019,5,2019-01-27,2019-02-02,1,438,47,704,JAN4
5,2019,6,2019-02-03,2019-02-09,2,443,41,587,FEV1
6,2019,7,2019-02-10,2019-02-16,2,406,41,567,FEV2
7,2019,8,2019-02-17,2019-02-23,2,526,32,568,FEV3
8,2019,9,2019-02-24,2019-03-02,2,448,30,600,FEV4
9,2019,10,2019-03-03,2019-03-09,3,543,45,318,MAR1
